In [ ]:
# Import packages
import numpy as np
import pandas as pd
from functions_fcsuml import *

# Load data
df = pd.read_parquet('data_exjobb_070425.parquet')
df.head()

In [ ]:
# Split into targets and data, and remove undesired features.
dataframe = df.copy()
Targets = pd.DataFrame(dataframe['LogAdjSalePrice202006'])
to_drop = ['TransactionId', 'BaseAreaName', 'DesoArea', 'geometry', 'DistAnyCity', 'DistAnyWater', 'SalePrice', 'LogAdjSalePrice202006', 'LogSalePrice',
            'AdjSalePrice202006']
dataframe = dataframe.drop(to_drop, axis=1)

In [ ]:
# Define prepare_data function
def prepare_data(dataframe_original):
    """Takes a dataframe and finds the columns with values that are non-numerical and assigns each unique non-numerical value a numerical value."""
    dataframe = dataframe_original.copy()
    # rows = np.shape(dataframe)[0]
    cols = np.shape(dataframe)[1]
    # print(type(dataframe))
    # print(cols, rows)

    for col in range(cols):
        c = type(dataframe.iloc[0, col]) # Value of the first element in column i row 1
        # print(c)
        if c != np.float64 and c != np.int64:
            a = dataframe.iloc[:, col].unique()   # Gives the unique values of a given column
            number_of_unique_values = np.shape(a)[0]
            b = np.linspace(1, number_of_unique_values, number_of_unique_values) / number_of_unique_values
            # a.reshape(len(a),1)
            # print(np.shape(a), a, "a[0]", a[0], type(a[0]))
            
            changed_data = pd.DataFrame(dataframe[f'{dataframe.axes[1][col]}'].copy())
            # print("changed_data", changed_data, " dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            # print("changed data", type(changed_data))
            for j in range(number_of_unique_values):
                # print("col",col)
                # print(dataframe[f'{dataframe.axes[1][col]}'])
                # changed_data = dataframe[f'{dataframe.axes[1][col]}'].copy()
                
                # print(j, "type", type(a[j]), type(b[j]))
                # print("value", a[j], b[j])
                if a[j] == None:
                    changed_data = changed_data.replace('None', b[j])
                else:
                    changed_data = changed_data.replace(a[j], b[j], regex=False)
                    
                    
            # print("changed data", changed_data)
            dataframe[f'{dataframe.axes[1][col]}'] = changed_data
            # print("dataframe", dataframe[f'{dataframe.axes[1][col]}'])
            
            

    return dataframe.fillna(0) # Returns the changed data with NaN values changed to zero.

In [ ]:
dataframe['EstimatedContractDate'] = dataframe['EstimatedContractDate'].astype('int64') // 10**9

prepared_data = prepare_data(dataframe)
prepared_data.insert(0, "TransactionId", df['TransactionId'])
print(prepared_data.shape)

In [ ]:
# Chaning to numpy array
def dataframe_to_numpy(dataframe):
    dataframe = dataframe.fillna(0)
    data = dataframe.to_numpy()


    data, removed_indices = remove_non_numbers(data)
    print("Removed indices:", removed_indices)



    data = data.astype(float) # Changing from type object to float so that numpy functions work properly.
    return data

Transaction_ID = df.iloc[:, 0]
# Transaction_ID = Transaction_ID.to_numpy()
# Transaction_ID = Transaction_ID.astype(np.float64)
# print(f'ID0: {Transaction_ID}, type: ')
# df = df.drop("TransactionId", axis=1)

data = dataframe_to_numpy(df)
targets = df2.values# dataframe_to_numpy(df2)
# targets = targets.values # .reshape(np.shape(targets)[0], 1)


# Divide the data set into training and testing data

training_data, testing_data, training_targets, testing_targets = train_test_split(data, targets, test_size=0.2, random_state=42)

testing_data, validation_data, testing_targets, validation_targets = train_test_split(testing_data, testing_targets, test_size=0.5, random_state=42)

print('X1 shape: ', training_data.shape)
print('X2 shape: ', testing_data.shape)
print('X3 shape: ', validation_data.shape)
print('X4 shape: ', training_targets.shape)
print('X5 shape: ', testing_targets.shape)
print('X6 shape: ', validation_targets.shape)

# datapoints = 5000
# training_data, testing_data, training_targets, testing_targets, validation_data, validation_targets = training_data[0:datapoints, :], testing_data[0:datapoints, :], training_targets[0:datapoints, :], testing_targets[0:datapoints, :], validation_data[0:datapoints, :], validation_targets[0:datapoints, :]

In [ ]:
# Normalizing the data
norm_data, normalization_variables = normalize(training_data)
norm_targets, max_targets, min_targets = normalize_feature(training_targets)

norm_testing_targets = (testing_targets-min_targets)/(max_targets-min_targets)
norm_validation_targets = (validation_targets-min_targets)/(max_targets-min_targets)
norm_validation_data = (validation_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
norm_testing_data = (testing_data-normalization_variables[1])/(normalization_variables[0]-normalization_variables[1])
print(np.shape(norm_data))

In [ ]:


name_list = ['norm_data_260303.parquet', 'norm_targets_260303.parquet', 'norm_testing_targets_260303.parquet', 'norm_validation_targets_260303.parquet',
             'norm_validation_data_260303.parquet', 'norm_testing_data_260303.parquet']

data_list = [norm_data, norm_targets, norm_testing_targets, norm_validation_targets, norm_validation_data, norm_testing_data]

for i in range(len(name_list)):
    temp = pd.DataFrame(data_list[i])
    temp.to_parquet(name_list[i])
